# COE 2022 "GPRA Friendly" Fixed (1 run)

If you want to run this notebook, keep in mind that the library path from the FLORIS config file has to be changed accordingly to your new path

## Imports and environment set up

In [1]:
from pathlib import Path
from itertools import product

import pandas as pd
import matplotlib.pyplot as plt

from whale import Project
from whale.utilities import load_yaml

pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

%load_ext autoreload
%autoreload 2

# Function to Extract Results

In [2]:
def extract_results(project_name):  # Run the project and clean up the logging
    #project_name = "UMaine_catenary"
    library_path = Path("library").resolve()
    
    metrics_configuration = {
        "# Turbines": {"metric": "n_turbines", "kwargs": {}},
        "Turbine Rating (MW)": {"metric": "turbine_rating", "kwargs": {}},
        "Project Capacity (MW)": {"metric": "capacity", "kwargs": {"units": "mw"}},
        "# OSS": {"metric": "n_substations", "kwargs": {}},
        "Total Export Cable Length (km)": {"metric": "export_system_total_cable_length", "kwargs": {}},
        "Total Array Cable Length (km)": {"metric": "array_system_total_cable_length", "kwargs": {}},
        "CapEx ($)": {"metric": "capex", "kwargs": {}},
        "CapEx per kW ($/kW)": {"metric": "capex", "kwargs": {"per_capacity": "kw"}},
        "OpEx ($)": {"metric": "opex", "kwargs": {}},
        "OpEx per kW ($/kW)": {"metric": "opex", "kwargs": {"per_capacity": "kw"}},
        "AEP (MWh)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "aep": True, "with_losses": True}
        },
        "AEP per kW (MWh/kW)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "per_capacity": "kw", "aep": True, "with_losses": True}
        },
        "Net Capacity Factor Without Unmodeled Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net"}
        },
        "Net Capacity Factor With All Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net", "with_losses": True}
        },
        "Gross Capacity Factor (%)": {"metric": "capacity_factor", "kwargs": {"which": "gross"}},
        "Energy Availability (%)": {"metric": "availability", "kwargs": {"which": "energy"}},
        "LCOE ($/MWh)": {"metric": "lcoe", "kwargs": {}},
        "IRR (%)": {"metric": "irr", "kwargs": {}},
        "NPV ($)": {"metric": "npv", "kwargs": {}},
    }

    metrics_order = [
        "# Turbines",
        "Turbine Rating (MW)",
        "Project Capacity (MW)",
        "# OSS",
        "Total Export Cable Length (km)",
        "Total Array Cable Length (km)",
        "FCR (%)",
        "Offtake Price ($/MWh)",
        "CapEx ($)",
        "CapEx per kW ($/kW)",
        "System CapEx for Export Cables ($)",
        "Installation CapEx for Export Cables ($)",
        "CapEx Without Export Cables ($)",
        "OpEx ($)",
        "OpEx per kW ($/kW)",
        "Annual OpEx per kW ($/kW)",
        "Energy Availability (%)",
        "Gross Capacity Factor (%)",
        "Net Capacity Factor Without Unmodeled Losses (%)",
        "Net Capacity Factor With All Losses (%)",
        "AEP (MWh)",
        "AEP per kW (MWh/kW)",
        "LCOE ($/MWh)",
        "IRR (%)",
        "NPV ($)",
    ]
    final_cols = ["CapEx ($)", "OpEx ($)", "Energy Production (GWh)", "Revenue ($)", "Cash Flow ($)"]
    
    config = load_yaml(
        library_path / "project/config",
        f"{project_name.replace(' ', '_')}_base.yaml"
    )
    project = Project(
        # Basic Model Configurations
        library_path=library_path,
        #weather_profile=library_path / "weather" / "ocean_wind_1_39.0_-74.0_1959_2023.csv",
        connect_floris_to_layout=True,
        connect_orbit_array_design=True,
        **config,
    )

    
    project.run(
        which_floris="wind_rose",
        full_wind_rose=False,
        floris_reinitialize_kwargs=dict(cut_in_wind_speed=3.0, cut_out_wind_speed=25.0)
    )
    project.wombat.env.cleanup_log_files()  # Delete logging data from WOMBAT
    

    
    #fig, ax = project.plot_farm(return_fig=True)
    print("Monopile cost and transition piece costs")
    print(project.orbit.phases['MonopileDesign'].detailed_output)
 
    df_capex  = pd.DataFrame(project.orbit.capex_breakdown.items(), columns=['CapEx Component', 'Value ($)'])

    df_capex["Value ($/kW)"] = df_capex["Value ($)"]/(project.capacity("mw")*1000)

    display(df_capex)
    
    df_capex.to_csv('library/results/' + project_name + '_CapEx_Breakdown.csv')
    
    df_opex = project.wombat.metrics.opex(frequency="annual", by_category = True)
    
    display(df_opex)
    
    df_opex.to_csv('library/results/' + project_name + '_OpEx_Breakdown.csv')
    
    list_of_dict = project.orbit.actions
    df = pd.DataFrame(list_of_dict)
    df.to_csv('library/results/' + project_name + '_Install_Sequence.csv')
    display(df)
    
    # Gather the high-level results
    report_df = project.generate_report(metrics_configuration, project_name).T
    export_system = project.orbit.system_costs["ExportCableInstallation"]
    export_installation = project.orbit.installation_costs["ExportCableInstallation"]
    capex_sans_export = project.orbit.total_capex - export_system - export_installation
    additional_reporting = pd.DataFrame(
        [
            ["FCR (%)", project.fixed_charge_rate],
            ["Offtake Price ($/MWh)", project.offtake_price],
            ["System CapEx for Export Cables ($)", export_system],
            ["Installation CapEx for Export Cables ($)", export_installation],
            ["CapEx Without Export Cables ($)", capex_sans_export],
            ["Annual OpEx per kW ($/kW)", report_df.loc["OpEx per kW ($/kW)", project_name] / project.operations_years],
        ],
        columns=["Project"] + report_df.columns.tolist(),
    ).set_index("Project")
    report_df = pd.concat((report_df, additional_reporting), axis=0).loc[metrics_order]

    # Gather the detailed results
    monthly_results = project.cash_flow(breakdown=True).join(project.energy_production(frequency="month-year")).fillna(0)
    monthly_results = monthly_results.assign(
        CapEx_Installation=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("Installation")]].sum(axis=1),
        CapEx_System=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("System")]].sum(axis=1),
    )

    # monthly_results.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_monthly_detailed_results.csv")
    monthly_results["CapEx ($)"] = monthly_results[["CapEx_Installation", "CapEx_Soft", "CapEx_Project", "CapEx_System", "CapEx_Turbine"]].sum(axis=1)
    monthly_results = monthly_results.rename(columns={"OpEx": "OpEx ($)","Revenue": "Revenue ($)", "cash_flow": "Cash Flow ($)"})[final_cols]

    if "ExportSystemDesign" in project.orbit._phases:
        export = "ExportSystemDesign"
    else:
        export = "ElectricalDesign"

    # Create the inputs data
    inputs = pd.DataFrame(
        [
            ["FCR", project.fixed_charge_rate],
            ["Discount rate (%)", project.discount_rate],
            ["# Turbines", project.n_turbines()],
            ["Turbine Rating (MW)", project.turbine_rating()],
            ["Project Capacity (MW)", project.capacity("mw")],
            ["# OSS", project.n_substations()],
            ["Substructure type", "??"],
            ["Row spacing (rotor diameters)", "not used for custom layouts"],
            ["Turbine spacing (rotor diameters)", "not used for custom layouts"],
            ["Depth (m)", project.orbit.config["site"]["depth"]],
            [
                "Mean wind speed (m/s)",
                project.weather.loc[
                    project.orbit_start_date: project.wombat.env.end_datetime,
                    "windspeed_100m"
                ].mean()
            ],
            ["Distance to landfall (km)", project.orbit.config["site"]["distance_to_landfall"]],
            ["Distance to port (km)", project.wombat.config.port_distance],
            ["Interconnection distance (km)", project.orbit._phases[export]._distance_to_interconnection],
            ["# of POIs", "??"],
            ["Export cable type", [*project.orbit._phases[export].cable_lengths_by_type]],
            ["Array cable type", [*project.orbit._phases["CustomArraySystemDesign"].cable_lengths_by_type]],
        ],
        columns=["Inputs", f"{project_name}"]
    ).set_index("Inputs")

    # Save the outputs
    report_df.index.name = "Metrics"
    wrong_indexes = ["Offtake Price ($/MWh)", "System CapEx for Export Cables ($)", "Installation CapEx for Export Cables ($)", "CapEx Without Export Cables ($)", "IRR (%)", "NPV ($)"]
    
    
    return report_df.drop(wrong_indexes,axis=0)

# Extract CapEx, OpEx, and AEP Outputs - Multiple Runs
Availability and OpEx varies depending on the run as it uses probabilistic functions, so we collect the data from x runs and calculate the average OpEx and AEP

In [3]:
def display_results_all_runs(n_runs):
    import pandas as pd
    
    df_list_fixed_bottom = list()
    df_list_floating = list()
    df_final_list = list()
    
    for i in range(0,n_runs):
        df_2 = extract_results("COE_2022_GF_fixed_bottom")
        
        
        if i == 0:
            df_list_floating = df_2
           
        else:
            df_list_floating = pd.concat([df_list_floating, df_2], axis=1)
                

            
    final_df = df_list_floating
                          
    return final_df

import time
t1 = time.perf_counter()

summary_results = display_results_all_runs(1)

t2 = time.perf_counter()
print('time taken to run:',t2-t1)

ORBIT library intialized at 'C:\COWER\WHaLE\examples\library'


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\COWER\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Monopile cost and transition piece costs
{'total_monopile_mass': 64280.760837733535, 'total_monopile_cost': 192842282.5132006, 'total_transition_piece_mass': 39913.62045552171, 'total_transition_piece_cost': 119740861.36656514}


,CapEx Component,Value ($),Value ($/kW)
0,Array System,"111,193,235.7130",185.3221
1,Export System,"100,357,800.0000",167.2630
2,Offshore Substation,"99,479,100.0000",165.7985
3,Scour Protection,"10,408,000.0000",17.3467
4,Substructure,"312,583,143.8798",520.9719
5,Array System Installation,"109,372,410.6504",182.2874
6,Export System Installation,"133,541,079.3068",222.5685
7,Offshore Substation Installation,"7,438,142.1148",12.3969
8,Scour Protection Installation,"41,331,879.6146",68.8865
9,Substructure Installation,"41,978,728.3421",69.9645


,operations,port_fees,equipment_cost,total_labor_cost,materials_cost,OpEx
year,,,,,,
2000,"3,946,783.5616",0,"25,990,726.8231",0,"767,504.0000","30,705,014.3848"
2001,"3,936,000.0000",0,"27,835,731.7107",0,"586,392.0000","32,358,123.7107"
2002,"3,936,000.0000",0,"67,106,521.9418",0,"1,308,624.0000","72,351,145.9418"
2003,"3,936,000.0000",0,"50,808,371.9981",0,"1,461,400.0000","56,205,771.9981"
2004,"3,946,783.5616",0,"44,988,000.6839",0,"1,145,440.0000","50,080,224.2455"
2005,"3,936,000.0000",0,"49,084,779.2512",0,"1,048,872.0000","54,069,651.2512"
2006,"3,936,000.0000",0,"49,829,823.7830",0,"1,129,712.0000","54,895,535.7830"
2007,"3,936,000.0000",0,"26,274,564.8714",0,"711,192.0000","30,921,756.8714"
2008,"3,946,783.5616",0,"27,787,977.9829",0,"656,320.0000","32,391,081.5446"


,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,max_waveheight,max_windspeed,transit_speed,site_depth,hub_height,per_trip
0,0.5000,Array Cable Installation Vessel,Mobilize,72.0000,"540,000.0000",ACTION,0.0000,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5000,Export Cable Installation Vessel,Mobilize,72.0000,"540,000.0000",ACTION,0.0000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.5000,Export Cable Burial Vessel,Mobilize,72.0000,"540,000.0000",ACTION,0.0000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Onshore Construction,Onshore Construction,0.0000,"110,325,876.2831",ACTION,0.0000,ExportCableInstallation,Landfall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.5000,Heavy Lift Vessel,Mobilize,72.0000,"975,000.0000",ACTION,0.0000,OffshoreSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3717,NaN,WTIV,Attach Blade,3.5000,"34,125.0000",ACTION,"6,241.6326",TurbineInstallation,NaN,TurbineInstallation,NaN,NaN,NaN,34.0000,136.0000,NaN
3718,NaN,WTIV,Release Blade,1.0000,"9,750.0000",ACTION,"6,242.6326",TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3719,NaN,WTIV,Lift Blade,1.3600,"13,260.0000",ACTION,"6,243.9926",TurbineInstallation,NaN,TurbineInstallation,NaN,NaN,NaN,34.0000,136.0000,NaN
3720,NaN,WTIV,Attach Blade,3.5000,"34,125.0000",ACTION,"6,247.4926",TurbineInstallation,NaN,TurbineInstallation,NaN,NaN,NaN,34.0000,136.0000,NaN


time taken to run: 319.48458799999935


In [4]:
display(summary_results)

,COE_2022_GF_fixed_bottom
Metrics,
# Turbines,50.0000
Turbine Rating (MW),12.0000
Project Capacity (MW),600.0000
# OSS,1.0000
Total Export Cable Length (km),118.0680
Total Array Cable Length (km),277.9831
FCR (%),0.0648
CapEx ($),"2,518,947,926.3953"
CapEx per kW ($/kW),"4,198.2465"


In [5]:
project_name = "COE_2022_GF_fixed_bottom"
summary_results.to_csv('library/results/' + project_name + '_LCOE_Breakdown.csv')

In [10]:
from construction_finance_param import con_fin_params

In [15]:
bos = 1286110371.15

turbine_capex = 1020000000.0000	

orbit_install_capex = 402207657.08

plant_cap = 600000



fixed = con_fin_params(bos, turbine_capex, orbit_install_capex, plant_cap, per_kW=False)
print(fixed)



{'soft_capex': 485791628.97058374, 'construction_insurance_capex': 26520269.268225, 'decomissioning_costs': 69380820.84629999, 'construction_financing': 115134221.83620875, 'procurement_contingency_costs': 109474406.059025, 'install_contingency_costs': 138761641.69259998, 'project_completion_capex': 26520269.268225}
